# Collaborative Filtering

This notebook implements collaborative filtering recommendation systems using **The Movies Dataset** (Kaggle - Rounak Banik):
- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering
- Matrix Factorization (SVD)
- Evaluate collaborative filtering performance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import models and evaluation from src
import sys
sys.path.append('..')
from src.models import UserBasedCF, ItemBasedCF, MatrixFactorizationCF
from src.evaluation import MetricsCalculator
from src.data import DataLoader
from src.data.data_preprocessor import DataPreprocessor

print("Modules loaded successfully!")

Modules loaded successfully!


import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')
from src.models import UserBasedCF, ItemBasedCF, MatrixFactorizationCF
from src.evaluation import MetricsCalculator
from src.data import DataLoader

print("Modules loaded successfully!")

In [2]:
# FIX #1 — Load from the correctly split preprocessed files
# (Run data_preprocessing.ipynb with method='user_based' first)
loader = DataLoader()
movies_metadata, train_ratings, test_ratings = loader.load_preprocessed_data()

# Enforce overlap
overlap_users = set(train_ratings['userId']) & set(test_ratings['userId'])
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)].copy()
test_ratings  = test_ratings[test_ratings['userId'].isin(overlap_users)].copy()

print(f"Overlap users:  {len(overlap_users):,}")
print(f"Train ratings:  {len(train_ratings):,}")
print(f"Test ratings:   {len(test_ratings):,}")

# VERIFICATION GATE #1
assert len(overlap_users) >= 95, f"Too few overlap users: {len(overlap_users)}"
print("✓ Verification gate: overlap users ≥ 95")

# ── FIX #2 — Prepare CF training data ───────────────────────────────────────
# With 256k overlap users, ALL train ratings belong to test users.
# Pre-select the 100 evaluation users (same random_state=42 as evaluation),
# then add 14,900 other users for density. This keeps cf_train ~1.2M rows
# instead of 20M (which causes OOM in similarity matrix construction).
rng = np.random.default_rng(42)

eval_users = MetricsCalculator._select_evaluation_users(
    test_ratings=test_ratings,
    train_ratings=train_ratings,
    n_users=100,
    random_state=42,
)
eval_user_set = set(int(u) for u in eval_users)

eval_user_train = train_ratings[train_ratings['userId'].isin(eval_user_set)]

other_train = train_ratings[~train_ratings['userId'].isin(eval_user_set)]
other_counts = other_train.groupby('userId').size()
other_eligible = other_counts[other_counts >= 3].index.tolist()
n_extra = min(14_900, len(other_eligible))
extra_users = set(rng.choice(other_eligible, size=n_extra, replace=False).tolist())

cf_train = pd.concat([
    eval_user_train,
    other_train[other_train['userId'].isin(extra_users)],
], ignore_index=True)

n_cf_ratings = len(cf_train)
print(f"\nCF training set:")
print(f"  Ratings: {n_cf_ratings:,}")
print(f"  Users:   {cf_train['userId'].nunique():,}")
print(f"  Movies:  {cf_train['movieId'].nunique():,}")

# VERIFICATION GATE #2: must have ≥ 15,000 ratings
assert n_cf_ratings >= 15_000, f"CF data too small: {n_cf_ratings:,} ratings"
print("✓ Verification gate: CF ratings ≥ 15,000")

Overlap users:  256,107
Train ratings:  20,881,168
Test ratings:   5,109,682
✓ Verification gate: overlap users ≥ 95



CF training set:
  Ratings: 1,198,121
  Users:   15,000
  Movies:  21,650
✓ Verification gate: CF ratings ≥ 15,000


## User-Based Collaborative Filtering

In [3]:
# FIX #2 — min_ratings=3, no arbitrary user caps.
# ItemBasedCF uses max_movies=3000 to keep the item×item similarity matrix
# at 3k×3k (72 MB) rather than 30k×30k (7.2 GB OOM).

print("Training User-Based CF (min_ratings=3, no user cap)...")
user_cf = UserBasedCF(
    cf_train,
    n_neighbors=50,
    min_ratings=3,
    max_users=None,
    max_movies=None,
)

print("\nTraining Item-Based CF (max_movies=3000 to prevent OOM)...")
item_cf = ItemBasedCF(
    cf_train,
    n_neighbors=50,
    min_ratings=3,
    max_users=None,
    max_movies=3000,
)

# Quick sanity check
test_user = list(eval_user_set)[0]
recs_ucf = user_cf.recommend(test_user, n=5)
recs_icf = item_cf.recommend(test_user, n=5)
print(f"\nSample user-CF recs for user {test_user}:")
print(recs_ucf.to_string(index=False) if not recs_ucf.empty else "  (no recs)")
print(f"\nSample item-CF recs for user {test_user}:")
print(recs_icf.to_string(index=False) if not recs_icf.empty else "  (no recs)")

Training User-Based CF (min_ratings=3, no user cap)...


User-based CF: 15000 users, 12498 movies, 1186390 ratings

Training Item-Based CF (max_movies=3000 to prevent OOM)...


Item-based CF: 14997 users, 3000 movies, 1033115 ratings



Sample user-CF recs for user 18432:
 movieId    score
     105 5.664143
     207 5.517406
    1209 5.318279
    5971 5.318279
    6807 5.318279

Sample item-CF recs for user 18432:
 movieId    score
    2019 4.447539
    1248 4.428452
     922 4.414184
    3677 4.396384
    7327 4.374804


## Item-Based Collaborative Filtering

In [4]:
# FIX #5 — Centered TruncatedSVD (replaces uncentered SVD treating missing=0)
# validate=True runs a grid search over n_components=[20,50,100] on a 20% val split
print("Training centered SVD Matrix Factorization (FIX #5)...")
mf_cf = MatrixFactorizationCF(
    cf_train,
    n_components=50,
    min_ratings=3,
    max_users=None,
    max_movies=None,
    random_state=42,
    validate=True,
)

print(f"\nValidation RMSE: {mf_cf.val_rmse:.4f}" if mf_cf.val_rmse else "")
print(f"Validation MAE:  {mf_cf.val_mae:.4f}"  if mf_cf.val_mae  else "")

# VERIFICATION GATE #5
if mf_cf.val_rmse is not None:
    assert mf_cf.val_rmse < 1.0, f"RMSE too high: {mf_cf.val_rmse:.4f}"
    print("✓ Verification gate: val_RMSE < 1.0")

recs_mf = mf_cf.recommend(test_user, n=5)
print(f"\nSample MF recs for user {test_user}:")
print(recs_mf.to_string(index=False))

Training centered SVD Matrix Factorization (FIX #5)...


SVD validation grid search:


  n_components=20: val_RMSE=0.9225, val_MAE=0.7086


  n_components=50: val_RMSE=0.9247, val_MAE=0.7104


  n_components=100: val_RMSE=0.9304, val_MAE=0.7152
Selected n_components=20 (val_RMSE=0.9225)


Matrix Factorization (centered SVD n=20): 15000 users, 12498 movies, 1186390 ratings | val_RMSE=0.8601

Validation RMSE: 0.8601
Validation MAE:  0.6533
✓ Verification gate: val_RMSE < 1.0

Sample MF recs for user 18432:
 movieId    score
    1206 4.130304
     924 4.106567
     904 4.100305
    1252 4.072986
     593 4.071980


## Matrix Factorization Collaborative Filtering

**DISABLED (old code — overwrites fixed mf_cf from cell-7)**

The old cell re-trained mf_cf with broken params: `MatrixFactorizationCF(train_ratings, n_components=50, n_epochs=30, min_ratings=5, max_users=500, max_movies=500)`. Replaced by centered TruncatedSVD in cell-7.

## Evaluate Collaborative Filtering Models

In [5]:
# Evaluate all CF models — SAME settings: K=10, threshold=4.0, 100 users, seed=42
calculator = MetricsCalculator()
eval_kwargs = dict(k=10, n_users=100, min_rating=4.0, random_state=42,
                   train_ratings=train_ratings)

print("Evaluating User-Based CF...")
user_cf_metrics = calculator.evaluate_model_with_recommendations(
    user_cf, test_ratings, **eval_kwargs)

print("Evaluating Item-Based CF...")
item_cf_metrics = calculator.evaluate_model_with_recommendations(
    item_cf, test_ratings, **eval_kwargs)

print("Evaluating Matrix Factorization CF...")
mf_cf_metrics = calculator.evaluate_model_with_recommendations(
    mf_cf, test_ratings, **eval_kwargs)

for name, m in [("User-Based CF", user_cf_metrics),
                ("Item-Based CF", item_cf_metrics),
                ("Matrix Factorization", mf_cf_metrics)]:
    print(f"\n{name}:")
    for k, v in m.items():
        print(f"  {k}: {v:.4f}")

# VERIFICATION GATE #2: all models must evaluate on same user set
for name, m in [("User-CF", user_cf_metrics),
                ("Item-CF", item_cf_metrics),
                ("MF", mf_cf_metrics)]:
    assert m['evaluated_users'] >= 90, \
        f"{name}: only {m['evaluated_users']} users evaluated (need ≥90)"
print("\n✓ Verification gate: all CF models evaluated ≥ 90 users")

Evaluating User-Based CF...


Evaluating Item-Based CF...


Evaluating Matrix Factorization CF...



User-Based CF:
  precision@k: 0.0012
  recall@k: 0.0004
  f1@k: 0.0006
  hit_rate: 0.0010
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0251

Item-Based CF:
  precision@k: 0.0357
  recall@k: 0.0333
  f1@k: 0.0345
  hit_rate: 0.0300
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0099

Matrix Factorization:
  precision@k: 0.1000
  recall@k: 0.0787
  f1@k: 0.0881
  hit_rate: 0.0840
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0051

✓ Verification gate: all CF models evaluated ≥ 90 users


## Save Collaborative Filtering Metrics

In [6]:
cf_metrics = {
    'user_based': user_cf_metrics,
    'item_based': item_cf_metrics,
    'matrix_factorization': mf_cf_metrics,
}
calculator.save_metrics(cf_metrics, '../reports/collaborative_filtering_metrics.json')

# SVD validation table
print("\nSVD Validation Table:")
print(f"{'n_components':>12} | {'val_RMSE':>9} | {'val_MAE':>8}")
print("-" * 38)
if mf_cf.val_rmse:
    print(f"{mf_cf.n_components:>12} | {mf_cf.val_rmse:>9.4f} | {mf_cf.val_mae:>8.4f}")

print("\nCollaborative Filtering metrics saved → reports/collaborative_filtering_metrics.json")

# Summary
print("\nCollaborative Filtering Summary:")
for name, m in cf_metrics.items():
    print(f"  {name}: P@10={m['precision@k']:.4f}  R@10={m['recall@k']:.4f}  "
          f"F1={m['f1@k']:.4f}  Users={m['evaluated_users']}")


SVD Validation Table:
n_components |  val_RMSE |  val_MAE
--------------------------------------
          20 |    0.8601 |   0.6533

Collaborative Filtering metrics saved → reports/collaborative_filtering_metrics.json

Collaborative Filtering Summary:
  user_based: P@10=0.0012  R@10=0.0004  F1=0.0006  Users=100
  item_based: P@10=0.0357  R@10=0.0333  F1=0.0345  Users=100
  matrix_factorization: P@10=0.1000  R@10=0.0787  F1=0.0881  Users=100


## Summary

In [7]:
print("Collaborative Filtering Summary:")
print(f"\nUser-Based CF:")
for metric, value in user_cf_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nItem-Based CF:")
for metric, value in item_cf_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nMatrix Factorization CF:")
for metric, value in mf_cf_metrics.items():
    print(f"  {metric}: {value:.4f}")

print(f"\nBest performing model based on F1 score:")
best_model = max(['user_based', 'item_based', 'matrix_factorization'], 
                key=lambda x: cf_metrics[x]['f1@k'])
print(f"  {best_model}: {cf_metrics[best_model]['f1@k']:.4f}")

Collaborative Filtering Summary:

User-Based CF:
  precision@k: 0.0012
  recall@k: 0.0004
  f1@k: 0.0006
  hit_rate: 0.0010
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0251

Item-Based CF:
  precision@k: 0.0357
  recall@k: 0.0333
  f1@k: 0.0345
  hit_rate: 0.0300
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0099

Matrix Factorization CF:
  precision@k: 0.1000
  recall@k: 0.0787
  f1@k: 0.0881
  hit_rate: 0.0840
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0051

Best performing model based on F1 score:
  matrix_factorization: 0.0881
